<a href="https://colab.research.google.com/github/TPS-Projects/Colab/blob/main/Extra%C3%A7%C3%A3o_de_Extratos_Crehnor_Cresol.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
# Apagar Pastas
# !rm -rf /content/_ /

# Apagar arquivos da pasta
'''
!rm -rf /content/_Cresol/*
!rm -rf /content/_Crehnor/*


In [2]:
#Criar Pastas
'''
import os

pasta1 = '/content/_Crehnor'
pasta2 = '/content/_Cresol'

os.makedirs(pasta1, exist_ok=True)
os.makedirs(pasta2, exist_ok=True)

# Verifica se ambas foram criadas e exibe a mensagem correspondente
if os.path.isdir(pasta1) and os.path.isdir(pasta2):
    print(f"As pastas {pasta1}\n- {pasta2} foram criadas:\n- Faça o uploado dos arquivos em pdf")
else:
    print("Erro ao criar uma ou mais pastas.")

As pastas /content/_Crehnor
- /content/_Cresol foram criadas:
- Faça o uploado dos arquivos em pdf


In [ ]:
#instalar
!pip install pdfplumber
!apt-get update -qq
!apt-get install poppler-utils tesseract-ocr tesseract-ocr-por -qq
!pip install pdf2image pytesseract pandas openpyxl -q

In [86]:
#Extração CREHNOR final

import os
import re
import pdfplumber
import pandas as pd


# 2. COLE O CAMINHO DA SUA PASTA AQUI
pasta_pdfs = "/content/_Crehnor"

dados_extraidos = []

print("Iniciando a leitura dos PDFs...")

for arquivo in os.listdir(pasta_pdfs):
    if arquivo.lower().endswith(".pdf"):
        caminho_completo = os.path.join(pasta_pdfs, arquivo)
        print(f"Lendo: {arquivo}")

    try:
            with pdfplumber.open(caminho_completo) as pdf:
                texto = ""
                for pagina in pdf.pages:
                    texto += pagina.extract_text() + "\n"

                dados = {
                    "Arquivo": arquivo,
                    "Agência": None,
                    "Conta": None,
                    "Cliente/Nome": None,
                    "CNPJ": None,
                    "Saldo": None,
                    "Tipo Layout": None
                }

                # Layout 1: Conta Bloqueada / Corrente
                if "SALDO DISPONÍVEL" in texto:
                    dados["Tipo Layout"] = "Conta Corrente"

                    agencia = re.search(r"AGÊNCIA:\s*(.*)", texto)
                    conta = re.search(r"CONTA:\s*(\d+)", texto)
                    cliente = re.search(r"CLIENTE:\s*(.*)", texto)
                    cnpj = re.search(r"CPF/CNPJ:\s*([\d\.\-\/]+)", texto)

                    # REGEX ATUALIZADO: Aceita quebras de linha e barras invisíveis
                    saldo = re.search(r"SALDO DISPONÍVEL \(\=\)[\s\|]*(R\$\s*[\d\.,]+)", texto)

                    if agencia: dados["Agência"] = agencia.group(1).strip()
                    if conta: dados["Conta"] = conta.group(1).strip()
                    if cliente: dados["Cliente/Nome"] = cliente.group(1).strip()
                    if cnpj: dados["CNPJ"] = cnpj.group(1).strip()
                    if saldo: dados["Saldo"] = saldo.group(1).strip()

                # Layout 2: Extrato de Aplicação
                elif "AG/CONTA:" in texto and "NOME:" in texto:
                    dados["Tipo Layout"] = "Aplicação"

                    ag_conta = re.search(r"AG/CONTA:\s*(\d+)/(\d+)", texto)
                    nome = re.search(r"NOME:\s*(.*)", texto)
                    cnpj = re.search(r"CNPJ:\s*([\d\.\-\/]+)", texto)

                    # REGEX ATUALIZADO: Aceita quebras de linha e barras invisíveis
                    total = re.search(r"Total[\s\|]*([\d\.,]+[A-Z]?)", texto)

                    if ag_conta:
                        dados["Agência"] = ag_conta.group(1).strip()
                        dados["Conta"] = ag_conta.group(2).strip()
                    if nome: dados["Cliente/Nome"] = nome.group(1).strip()
                    if cnpj: dados["CNPJ"] = cnpj.group(1).strip()
                    if total: dados["Saldo"] = total.group(1).strip()

                dados_extraidos.append(dados)
    except Exception as e:
      print(f"Erro ao ler o arquivo {arquivo}: {e}")

# 3. Gerar a planilha Excel
df = pd.DataFrame(dados_extraidos)
caminho_saida = "/content/Extratos_CREHNOR.xlsx"
df.to_excel(caminho_saida, index=False)

print("-" * 30)
print(f"Extração concluída com sucesso!")
print(f"Sua tabela foi salva em: {caminho_saida}")

Iniciando a leitura dos PDFs...
------------------------------
Extração concluída com sucesso!
Sua tabela foi salva em: /content/Extratos_CREHNOR.xlsx


In [89]:
#Estração Cresol
import os
import re
import pandas as pd
from pdf2image import convert_from_path
import pytesseract

# Nome da Pasta já criada
pasta_pdfs = "/content/_Cresol"

dados_extraidos = []

print("Iniciando a extração visual (OCR) dos extratos Cresol...")

for arquivo in os.listdir(pasta_pdfs):
    # Nível 1: Dentro do For (4 espaços)
    if arquivo.lower().endswith(".pdf"):
        caminho_completo = os.path.join(pasta_pdfs, arquivo)

        # Nível 2: O Try fica alinhado aqui (8 espaços)
        try:
            # Converte o PDF em imagem (Ajuste Dpi para melhor resultado)
            imagens = convert_from_path(caminho_completo, dpi=500)
            texto = ""

            # Lê o texto da imagem usando o idioma português ("por")
            for img in imagens:
                texto += pytesseract.image_to_string(img, lang="por") + "\n"

            # Processa apenas Cresol
            if "EXTRATO CONSOLIDADO DE POUPANÇA" in texto.upper() or "EXTRATO CONSOLIDADO DE CONTA" in texto.upper():
                print(f"Lendo com OCR: {arquivo}")

                dados = {
                    "Arquivo": arquivo,
                    "Agência": None,
                    "Conta": None,
                    "Cliente/Nome": None,
                    "CNPJ": None,
                    "Saldo": None,
                    "Tipo Layout": None
                }

                # LAYOUT A: Poupança
                if "POUPANÇA" in texto.upper() or "POUPANCA" in texto.upper():
                    dados["Tipo Layout"] = "Poupança"

                    # Agência
                    agencia = re.search(r"AG[EÊ]NCIA:\s*(.*)", texto, re.IGNORECASE)
                    if agencia:
                        dados["Agência"] = agencia.group(1).split("FONE")[0].strip()

                    # Conta e Nome
                    conta_nome = re.search(r"CONTA:\s*([\d\.\-]+)\s+(.*)", texto, re.IGNORECASE)
                    if conta_nome:
                        dados["Conta"] = conta_nome.group(1).strip()
                        dados["Cliente/Nome"] = conta_nome.group(2).strip()

                    # CNPJ (Busca Robusta OCR)
                    cnpj_sujo = re.search(r"(\d{2}[\.\,\s]*\d{3}[\.\,\s]*\d{3}[\/\s\|lI]*\d{4}[\-\s]*\d{2})", texto)
                    if cnpj_sujo:
                        numeros_cnpj = re.sub(r"\D", "", cnpj_sujo.group(1))
                        if len(numeros_cnpj) == 14:
                            dados["CNPJ"] = f"{numeros_cnpj[:2]}.{numeros_cnpj[2:5]}.{numeros_cnpj[5:8]}/{numeros_cnpj[8:12]}-{numeros_cnpj[12:]}"
                        else:
                            dados["CNPJ"] = cnpj_sujo.group(1).strip()

                    # Saldo
                    saldo = re.search(r"SALDO\s*TOTAL[^\d]*([\d\.,]+\s*[CD]?)", texto, re.IGNORECASE)
                    if saldo:
                        dados["Saldo"] = saldo.group(1).strip()

                # LAYOUT B: Conta Corrente
                elif "CONTA CORRENTE" in texto.upper():
                    dados["Tipo Layout"] = "Conta Corrente"

                    # Conta e Nome
                    conta_nome = re.search(r"([\d\.]+\-\d{1,2})[\s\-]+([A-Z][A-Z\s]+)", texto)
                    if conta_nome:
                        dados["Conta"] = conta_nome.group(1).strip()
                        dados["Cliente/Nome"] = conta_nome.group(2).strip()

                    # Agência
                    agencia = re.search(r"EXTRATO CONSOLIDADO DE CONTA CORRENTE[\s\n]*(\d{4}\-\d\s+[A-Z\s]+)", texto, re.IGNORECASE)
                    if agencia:
                        dados["Agência"] = agencia.group(1).strip()

                    # CNPJ (Busca Robusta OCR)
                    cnpj_sujo = re.search(r"(\d{2}[\.\,\s]*\d{3}[\.\,\s]*\d{3}[\/\s\|lI]*\d{4}[\-\s]*\d{2})", texto)
                    if cnpj_sujo:
                        numeros_cnpj = re.sub(r"\D", "", cnpj_sujo.group(1))
                        if len(numeros_cnpj) == 14:
                            dados["CNPJ"] = f"{numeros_cnpj[:2]}.{numeros_cnpj[2:5]}.{numeros_cnpj[5:8]}/{numeros_cnpj[8:12]}-{numeros_cnpj[12:]}"
                        else:
                            dados["CNPJ"] = cnpj_sujo.group(1).strip()

                    # Saldo
                    saldo = re.search(r"SALDO\s*TOTAL[^\d]*([\d\.,]+\s*[CD]?)", texto, re.IGNORECASE)
                    if saldo:
                        dados["Saldo"] = saldo.group(1).strip()

                # Anexa os dados extraídos (ainda dentro do if do extrato)
                dados_extraidos.append(dados)

        # Nível 2: O Except fica EXATAMENTE na mesma reta que o Try (8 espaços)
        except Exception as e:
            print(f"Erro ao ler o arquivo {arquivo} com OCR: {e}")

# 3. Gerar a planilha Excel dedicada (Sem espaços, colado na esquerda)
df = pd.DataFrame(dados_extraidos)
caminho_saida = "/content/Extratos_Cresol.xlsx"
df.to_excel(caminho_saida, index=False)

print("-" * 30)
print(f"Extração visual concluída com sucesso!")
print(f"Sua tabela gerada por OCR foi salva em: {caminho_saida}")

Iniciando a extração visual (OCR) dos extratos Cresol...
Lendo com OCR: 069.524-6 - x.pdf
Lendo com OCR: 069.524-6.pdf
Lendo com OCR: 078.235-1.pdf
Lendo com OCR: 065.642-9.pdf
Lendo com OCR: 078.235-1 - x.pdf
Lendo com OCR: 065.642-9 - x.pdf
------------------------------
Extração visual concluída com sucesso!
Sua tabela gerada por OCR foi salva em: /content/Extratos_Cresol.xlsx


In [ ]:
#teste normalizar e ocerizar antes sem ganho
import os
import re
import pandas as pd

from pdf2image import convert_from_path
import pytesseract
from PIL import ImageOps

# 1. CONFIGURAÇÕES
pasta_pdfs = "/content/_Cresol"
caminho_saida = "/content/Extratos_Cresol.xlsx"

dados_extraidos = []

# 2. NORMALIZAR TEXTO
def normalizar_texto(texto):
    texto = texto.replace("|", "I")
    texto = texto.replace("’", "'")
    texto = texto.replace("“", '"')
    texto = texto.replace("”", '"')
    texto = re.sub(r"[ ]+", " ", texto)
    texto = re.sub(r"\n+", "\n", texto)
    return texto


# 3. OCR DO PDF
def extrair_com_ocr(pdf):
    imagens = convert_from_path(pdf, dpi=400)
    texto = ""
    config = r'--oem 3 --psm 6'
    for img in imagens:
        img = ImageOps.grayscale(img)
        img = ImageOps.autocontrast(img)
        texto += pytesseract.image_to_string(
            img,
            lang="por",
            config=config
        )
        texto += "\n"
    return texto


# 4. PERCORRER OS PDFs
for arquivo in os.listdir(pasta_pdfs):
    if not arquivo.lower().endswith(".pdf"):
        continue
    caminho_pdf = os.path.join(pasta_pdfs, arquivo)

    try:
        print(f"Lendo arquivo: {arquivo}")

        # OCR
        texto = extrair_com_ocr(caminho_pdf)
        texto = normalizar_texto(texto)

        # Dicionário para armazenar os dados deste PDF
        dados = {}

        # Nome do arquivo
        dados["Arquivo"] = arquivo

        # LAYOUT A: POUPANÇA
        if "POUPANÇA" in texto.upper() or "POUPANCA" in texto.upper():
            dados["Tipo Layout"] = "Poupança"

            # Agência
            agencia = re.search(
                r"AG[EÊ]NCIA:\s*(.*)",
                texto,
                re.IGNORECASE
            )

            if agencia:
                dados["Agência"] = (
                    agencia.group(1)
                    .split("FONE")[0]
                    .strip()
                )

            # Conta + Nome

            conta_nome = re.search(
                r"CONTA:\s*([\d\.\-]+)\s+(.*)",
                texto,
                re.IGNORECASE
            )

            if conta_nome:
                dados["Conta"] = conta_nome.group(1).strip()
                dados["Cliente/Nome"] = (
                    conta_nome.group(2).strip()
                )
            # CNPJ
            cnpj = re.search(
                r"(\d{2}\.\d{3}\.\d{3}/\d{4}\-\d{2})",
                texto
            )

            if cnpj:

                dados["CNPJ"] = cnpj.group(1).strip()

            # Saldo

            saldo = re.search(
                r"SALDO\s*TOTAL[^\d]*([\d\.,]+\s*[CD]?)",
                texto,
                re.IGNORECASE
            )

            if saldo:
                dados["Saldo"] = saldo.group(1).strip()


        # LAYOUT B: CONTA CORRENTE
        elif "CONTA CORRENTE" in texto.upper():
            dados["Tipo Layout"] = "Conta Corrente"

            # CNPJ
            cnpj = re.search(
                r"(\d{2}\.\d{3}\.\d{3}/\d{4}\-\d{2})",
                texto
            )

            # Conta + Nome
            conta_nome = re.search(
                r"([\d\.]+\-\d{1,2})[\s\-]+([A-Z][A-Z\s]+)",
                texto
            )

            # Agência
            agencia = re.search(
                r"EXTRATO CONSOLIDADO DE CONTA CORRENTE"
                r"[\s\n]*(\d{4}\-\d\s+[A-Z\s]+)",
                texto,
                re.IGNORECASE
            )

            # Saldo
            saldo = re.search(
                r"SALDO\s*TOTAL[^\d]*([\d\.,]+\s*[CD]?)",
                texto,
                re.IGNORECASE
            )

            # Salvar Agência
            if agencia:

                dados["Agência"] = (
                    agencia.group(1).strip()
                )

            # Salvar Conta e Cliente
            if conta_nome:

                dados["Conta"] = (
                    conta_nome.group(1).strip()
                )

                dados["Cliente/Nome"] = (
                    conta_nome.group(2).strip()
                )

            # Salvar CNPJ
            if cnpj:
                dados["CNPJ"] = (
                    cnpj.group(1).strip()
                )

            # Salvar Saldo
            if saldo:
                dados["Saldo"] = (
                    saldo.group(1).strip()
                )

        # LAYOUT NÃO IDENTIFICADO
        else:
            dados["Tipo Layout"] = "Não identificado"

        # ADICIONAR RESULTADO
        dados_extraidos.append(dados)
        print(f"OK: {arquivo}")

    # TRATAMENTO DE ERRO
    except Exception as e:
        print(
            f"Erro ao ler o arquivo {arquivo} "
            f"com OCR: {e}"
        )

# 5. GERAR PLANILHA EXCEL
df = pd.DataFrame(dados_extraidos)
df.to_excel(
    caminho_saida,
    index=False
)

# 6. RESULTADO
print("-" * 30)
print(
    "Extração visual concluída com sucesso!"
)
print(
    f"Sua tabela gerada por OCR foi salva em: "
    f"{caminho_saida}"
)

Lendo arquivo: 078.235-1 - x.pdf
OK: 078.235-1 - x.pdf
Lendo arquivo: 078.235-1.pdf
OK: 078.235-1.pdf
Lendo arquivo: 065.642-9.pdf
OK: 065.642-9.pdf
Lendo arquivo: 069.524-6 - x.pdf
OK: 069.524-6 - x.pdf
Lendo arquivo: 069.524-6.pdf
OK: 069.524-6.pdf
------------------------------
Extração visual concluída com sucesso!
Sua tabela gerada por OCR foi salva em: /content/Extratos_Cresol.xlsx
